# Persistent Agent Memory with Shodh-Memory

Agents built with the OpenAI Agents SDK forget everything between runs. This is fine for stateless tasks, but real-world agents need to remember user preferences, past decisions, learnings, and pending work across sessions.

This cookbook shows how to give your agents **persistent cognitive memory** using [Shodh-Memory](https://github.com/varun29ankuS/shodh-memory) — a local-first cognitive memory system that runs as a single binary with no external dependencies.

**What you will learn:**
1. Using `ShodhSession` for automatic conversation persistence (drop-in `Session` replacement)
2. Using `ShodhTools` for explicit memory operations (remember, recall, forget, todos)
3. Combining both for a fully memory-capable agent

**What makes this different from SQLite/Redis sessions:**
- **Cognitive retrieval** — recall memories by meaning via semantic search, not just keyword match
- **Knowledge graph** — entities and relationships are extracted and connected automatically with spreading activation
- **Biological memory dynamics** — memories strengthen with use (Hebbian learning) and fade naturally following decay curves from cognitive science (Wixted 2004)
- **Local-first** — runs entirely on your machine, no cloud API, no data leaves your hardware

## Prerequisites

### Step 0: API Keys

You need an OpenAI API key for the agent's LLM calls. Shodh-Memory runs locally and uses its own API key for authentication.

In [ ]:
import os
import uuid

# OpenAI API key for the agent's LLM
os.environ.setdefault("OPENAI_API_KEY", "your-openai-api-key")

# Shodh-Memory API key (local server authentication)
os.environ.setdefault("SHODH_API_KEY", "your-shodh-api-key")

# Unique run ID to isolate state per execution — avoids stale results on reruns.
RUN_ID = uuid.uuid4().hex[:8]
USER_ID = f"cookbook-user-{RUN_ID}"
print(f"Run ID: {RUN_ID}, User: {USER_ID}")

### Step 1: Install dependencies

In [ ]:
%pip install openai-agents shodh-memory[openai-agents]

### Step 2: Start the Shodh-Memory server

Shodh-Memory runs as a local HTTP server. Download the binary for your platform from the [releases page](https://github.com/varun29ankuS/shodh-memory/releases) or install via pip:

```bash
# Option A: Download binary (fastest, ~15MB)
# See https://github.com/varun29ankuS/shodh-memory/releases

# Option B: Install via pip (includes the binary)
pip install shodh-memory

# Start the server (runs on http://localhost:3030)
shodh-memory serve
```

The server should be running before proceeding. It stores data in `~/.shodh-memory/` by default.

In [ ]:
import requests

SHODH_SERVER = "http://localhost:3030"

try:
    resp = requests.get(f"{SHODH_SERVER}/health", timeout=5)
    print(f"Shodh-Memory server is running: {resp.json()}")
except requests.exceptions.ConnectionError:
    print("Shodh-Memory server is not running. Start it with: shodh-memory serve")

## Part 1: Session-Based Memory

The simplest integration — replace `SQLiteSession` or `RedisSession` with `ShodhSession`. The agent gets persistent conversation history that benefits from cognitive consolidation and semantic indexing.

In [ ]:
from agents import Agent, Runner
from shodh_memory.integrations.openai_agents import ShodhSession

# Create a session. Each session_id gets isolated conversation history.
session = ShodhSession(
    session_id=f"cookbook-session-{RUN_ID}",
    server_url=SHODH_SERVER,
    user_id=USER_ID,
    api_key=os.environ["SHODH_API_KEY"],
)

agent = Agent(
    name="Assistant",
    instructions="Reply concisely. You have persistent cognitive memory across sessions.",
)

# Clear for a clean demo.
await session.clear_session()

In [ ]:
# Turn 1: Tell the agent something.
result = await Runner.run(agent, "My name is Alice and I work on autonomous drones.", session=session)
print(f"Assistant: {result.final_output}")

In [ ]:
# Turn 2: The agent remembers.
result = await Runner.run(agent, "What's my name and what do I work on?", session=session)
print(f"Assistant: {result.final_output}")

In [ ]:
# Check what's stored in the session.
items = await session.get_items()
print(f"Session has {len(items)} items stored in shodh-memory.")
for i, item in enumerate(items, 1):
    print(f"  {i}. [{item.get('role', '?')}] {item.get('content', '')[:80]}")

## Part 2: Tool-Based Memory

For agents that need **explicit cognitive control** over what they remember and recall, use `ShodhTools`. This gives the agent 8 tools:

| Tool | Purpose |
|------|---------|
| `shodh_remember` | Store a memory with type and tags |
| `shodh_recall` | Cognitive retrieval via semantic search across all memories |
| `shodh_forget` | Delete outdated or incorrect memories |
| `shodh_context_summary` | Get recent decisions, learnings, context |
| `shodh_proactive_context` | Surface memories relevant to current topic |
| `shodh_add_todo` | Create tasks with priority and projects |
| `shodh_list_todos` | List pending tasks |
| `shodh_complete_todo` | Mark tasks complete |

In [ ]:
from shodh_memory.integrations.openai_agents import ShodhTools

tools = ShodhTools(
    server_url=SHODH_SERVER,
    user_id=USER_ID,
    api_key=os.environ["SHODH_API_KEY"],
)

memory_agent = Agent(
    name="Memory Agent",
    instructions=(
        "You have persistent cognitive memory. Use shodh_remember to store important information. "
        "Use shodh_recall to search your memories when the user asks about something. "
        "Use shodh_add_todo to track tasks. Always confirm what you did."
    ),
    tools=tools.as_list(),
)

print(f"Agent has {len(tools.as_list())} memory tools: {[t.name for t in tools.as_list()]}")

In [ ]:
# Store a preference.
result = await Runner.run(
    memory_agent,
    "Remember that I prefer Rust for backend services and Python for ML prototyping.",
)
print(f"Agent: {result.final_output}")

In [ ]:
# Recall it semantically — the query doesn't need to match exact words.
result = await Runner.run(
    memory_agent,
    "What programming languages does the user like for different tasks?",
)
print(f"Agent: {result.final_output}")

In [ ]:
# Create a task.
result = await Runner.run(
    memory_agent,
    "Add a high-priority todo: Review the navigation module for the drone project.",
)
print(f"Agent: {result.final_output}")

In [ ]:
# List pending tasks.
result = await Runner.run(memory_agent, "What tasks do I have pending?")
print(f"Agent: {result.final_output}")

## Part 3: Combining Session + Tools

The most powerful setup — the agent gets automatic conversation persistence **and** explicit cognitive memory tools. The session handles short-term context; the tools handle long-term knowledge with biological decay and Hebbian strengthening.

In [ ]:
full_session = ShodhSession(
    session_id=f"cookbook-full-{RUN_ID}",
    server_url=SHODH_SERVER,
    user_id=USER_ID,
    api_key=os.environ["SHODH_API_KEY"],
)

full_agent = Agent(
    name="Full Memory Agent",
    instructions=(
        "You are a personal assistant with persistent cognitive memory. "
        "Conversation history is saved automatically via your session. "
        "For important facts, decisions, or preferences, use shodh_remember to store them explicitly. "
        "Use shodh_recall when you need to search past knowledge. "
        "Use shodh_add_todo to track tasks the user mentions."
    ),
    tools=tools.as_list(),
)

await full_session.clear_session()

# Multi-turn conversation with both session memory and explicit cognitive tools.
messages = [
    "I'm building a drone delivery system. The max payload is 2.5kg and range is 15km.",
    "Remember those specs — they're critical for the route planning module.",
    "Add a todo: Test the obstacle avoidance with 20km/h crosswind, high priority.",
    "What are the drone specs I mentioned earlier?",
]

for msg in messages:
    print(f"User: {msg}")
    result = await Runner.run(full_agent, msg, session=full_session)
    print(f"Agent: {result.final_output}")
    print()

## How Cognitive Memory Works Under the Hood

Shodh-Memory is not a simple key-value store. It implements a biologically-inspired cognitive architecture:

1. **Storage**: Memories are embedded using MiniLM-L6 (384-dim vectors, runs locally via ONNX Runtime) and stored in RocksDB with a graph-based vector index (Vamana/SPANN).

2. **Entity Extraction**: Named entities are automatically extracted and linked into a knowledge graph with spreading activation. When the agent stores "Alice works on drones at Acme Corp", three entities (Alice, drones, Acme Corp) are created and connected.

3. **Cognitive Retrieval**: Recall uses a multi-stage pipeline — BM25 keyword search, vector similarity, entity matching, and knowledge graph traversal — fused via Reciprocal Rank Fusion.

4. **Hebbian Learning**: Every time a memory is accessed, its strength increases (additive boost). Memories that are never recalled gradually decay following a hybrid exponential/power-law curve based on Wixted (2004) — the same dynamics observed in human cognition.

5. **All Local**: The entire cognitive system runs as a single binary (~15MB). No Docker, no external database, no cloud API. Data stays on your machine.

## Cleanup

In [ ]:
# Clear all demo state for this run — sessions, memories, and todos.
await session.clear_session()
await full_session.clear_session()

# Also clean up tool-created memories/todos by deleting the run-scoped user.
import requests as _req
try:
    _req.delete(
        f"{SHODH_SERVER}/api/users/{USER_ID}",
        headers={"X-API-Key": os.environ["SHODH_API_KEY"]},
        timeout=10,
    )
except Exception:
    pass  # Best-effort cleanup.
print(f"Demo state for run {RUN_ID} cleaned up.")

## Next Steps

- **MCP Integration**: Shodh-Memory also runs as an MCP server with 45 cognitive memory tools, compatible with Claude, Cursor, and other MCP-aware clients.
- **LangChain & LlamaIndex**: Adapters available via `pip install shodh-memory[langchain]` and `pip install shodh-memory[llamaindex]`.
- **Edge Deployment**: The binary runs on Raspberry Pi, Jetson Nano, and other ARM64 devices — ideal for robotics and drone agents that need cognition at the edge.

Links:
- [GitHub Repository](https://github.com/varun29ankuS/shodh-memory)
- [Documentation](https://shodh-memory.com)
- [PyPI Package](https://pypi.org/project/shodh-memory/)